# 51 — n=2 top-up: 20 more pairs for the composition scaling arm

The scaling arm rests on 10 base pairs at n=2 but 30 groups at n=3 and
n=4. This notebook balances the design: 20 additional pairs sampled by
the same recipe as the original 10 (evenly spaced along the validated
pool's rho spectrum, each concept used at most twice, with the existing
pairs and their usage counts seeded in), then blend / vote / schedule at
lambda in {0, 0.4} on the new pairs only — settings identical to the
existing arms (K=4, steps=24, same prompts, angle metrics) so the records
merge seamlessly with 33/39/40. Figures live in notebook 52.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAMBDAS = (0.0, 0.4)
N_TOTAL_PAIRS = 30
MAX_USES_PER_CONCEPT = 2
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"
SMOKE = False  # True -> 2 new pairs x 2 prompts, artifacts get a _smoke suffix

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_NAMES = [r["synset"] for r in POOL["pool"]]
EXISTING = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())[
    "pairs"
]


def rho(a: str, b: str) -> float:
    key = f"{a}|{b}" if f"{a}|{b}" in POOL["pairwise_rho"] else f"{b}|{a}"
    return POOL["pairwise_rho"][key]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

SUFFIX = "_smoke" if SMOKE else ""
TOPUP_PATH = Path(f"resources/51_pairs_topup{SUFFIX}.json")
LOG_PATH = Path(f"resources/51_scaling2_topup{SUFFIX}.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"pool: {len(POOL_NAMES)} concepts, {len(EXISTING)} existing pairs")

Sampling: rebuild the 30-target even-rho grid over all pool pairs, let
each existing pair claim its nearest target, and fill the remaining 20
targets with the original nearest-pair-under-cap rule (usage counts
seeded from the existing pairs, duplicates excluded). Deterministic — no
RNG. Verified offline: the cap of 2 is feasible and the worst target
miss is 0.005 in rho.

In [ ]:
all_pairs = sorted(
    (
        {"a": a, "b": b, "pair_rho": rho(a, b)}
        for i, a in enumerate(POOL_NAMES)
        for b in POOL_NAMES[i + 1 :]
    ),
    key=lambda p: p["pair_rho"],
)
targets = [
    all_pairs[round(i * (len(all_pairs) - 1) / (N_TOTAL_PAIRS - 1))]["pair_rho"]
    for i in range(N_TOTAL_PAIRS)
]

taken = {(p["a"], p["b"]) for p in EXISTING}
uses = {}
for p in EXISTING:
    for s in (p["a"], p["b"]):
        uses[s] = uses.get(s, 0) + 1

claimed = set()
for p in EXISTING:
    nearest = min(
        (j for j in range(N_TOTAL_PAIRS) if j not in claimed),
        key=lambda j: abs(targets[j] - p["pair_rho"]),
    )
    claimed.add(nearest)

NEW_PAIRS = []
for j in range(N_TOTAL_PAIRS):
    if j in claimed:
        continue
    for p in sorted(all_pairs, key=lambda q: abs(q["pair_rho"] - targets[j])):
        key = (p["a"], p["b"])
        if key in taken:
            continue
        if uses.get(p["a"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        if uses.get(p["b"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        NEW_PAIRS.append(p)
        taken.add(key)
        uses[p["a"]] = uses.get(p["a"], 0) + 1
        uses[p["b"]] = uses.get(p["b"], 0) + 1
        break
    else:
        raise RuntimeError("usage cap prevented an even pair spread")

assert len(NEW_PAIRS) == N_TOTAL_PAIRS - len(EXISTING)
TOPUP_PATH.write_text(
    json.dumps(
        {
            "config": {
                "pool": "resources/32_concept_pool_bf16.json",
                "seeded_from": "resources/33_scaling_groups_bf16.json",
                "n_total_pairs": N_TOTAL_PAIRS,
                "max_uses_per_concept": MAX_USES_PER_CONCEPT,
            },
            "pairs": NEW_PAIRS,
        },
        indent=1,
    )
)
if SMOKE:
    NEW_PAIRS, QUESTIONS = NEW_PAIRS[:2], QUESTIONS[:2]
print(
    f"{len(NEW_PAIRS)} new pairs "
    f"(rho {NEW_PAIRS[0]['pair_rho']:.3f}..{NEW_PAIRS[-1]['pair_rho']:.3f}), "
    f"max concept uses {max(uses.values())}"
)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
# force the one-time cuDF WordNet parquet load (~300 MB on-GPU) now,
# while memory is free - inside the generation loop it can OOM
_ = harness.member_lemmas(NEW_PAIRS[0]["a"])
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted({s for p in NEW_PAIRS for s in (p["a"], p["b"])})
concept = {}
for name in used:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
print(f"{len(baseline_texts)} baselines, {len(concept)} concepts")

In [ ]:
for group in NEW_PAIRS:
    synsets = [group["a"], group["b"]]
    cs = [concept[s] for s in synsets]
    blend_guide = Concept.average(cs)
    for lam in LAMBDAS:
        methods = {
            # canonical wrapper has no fluency dial; unified loop w/ single guide
            "blend": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[blend_guide], k=K, steps=STEPS, fluency_weight=lam
            ),
            "vote": lambda: fur.generate_with_topk_multi_guide(
                prompts,
                guides=cs,
                k=K,
                steps=STEPS,
                normalize="zscore",
                fluency_weight=lam,
            ),
            "schedule": lambda: fur.generate_with_topk_multi_guide(
                prompts,
                guides=cs,
                k=K,
                steps=STEPS,
                scorer=RoundRobin(),
                fluency_weight=lam,
            ),
        }
        for method, generate in methods.items():
            torch.cuda.empty_cache()
            texts, probe = generate()
            harness.evaluate(
                prompts,
                texts,
                synsets,
                config={
                    "model": MODEL_KEY,
                    "method": method,
                    "n": 2,
                    "lam": lam,
                    **group,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts=dict(zip(synsets, cs)),
                angles=True,
            )
    print(f"rho={group['pair_rho']:.3f} {group['a']} / {group['b']}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == len(NEW_PAIRS) * len(LAMBDAS) * 3 * len(prompts)
assert all(
    r["expression"] is not None and r["angles"] is not None for r in parsed
), "metric v2 + v3 present everywhere"
assert not any(
    v != v
    for r in parsed
    for s in (r["config"]["a"], r["config"]["b"])
    for v in r["angles"][s]["delta"].values()
), "NaN in angle deltas"
print(f"gate: {len(parsed)} records parse, expression + angles logged")

In [ ]:
def counts(path, methods=("blend", "vote", "schedule")):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] in methods:
            rows.append((c.get("lam", 0.0), c["n"]))
    return rows


if SMOKE:
    print("SMOKE RUN — _smoke artifacts only; set SMOKE = False and rerun")
else:
    merged = (
        counts("resources/33_scaling3_bf16.jsonl")
        + counts("resources/39_scaling4.jsonl")
        + counts("resources/40_greedy_lambda.jsonl")
        + counts(LOG_PATH)
    )
    table = pd.Series(merged).value_counts().sort_index()
    print(table)
    assert all(table[(lam, n)] == 900 for lam in (0.0, 0.4) for n in (2, 3, 4))
    print("balanced: 900 records per (lambda, n) cell")